## This is a copy of measuring intersectionality construct models

I am just breaking it down to see the outcome of each step

In [3]:
#import the necessary libraries

#Data Wrangling
import pandas
import numpy as np
import string
import os
from nltk.tokenize import word_tokenize, sent_tokenize
from random import choices

import gensim #library needed for word2vec

In [4]:
def get_path(pathname):
    allFiles = os.listdir(pathname)
    allFiles = [pathname+file for file in allFiles]
    return(allFiles)

In [5]:
def fast_tokenize(text):
    
    # Get a list of punctuation marks
    punct = string.punctuation + '“' + '”' + '‘' + "’"
    
    lower_case = text.lower()
    lower_case = lower_case.replace('—', ' ').replace('\n', ' ')
    
    # Iterate through text removing punctuation characters
    no_punct = "".join([char for char in lower_case if char not in punct])
    
    # Split text over whitespace into list of words
    tokens = no_punct.split()
    
    return tokens

## 1. Import and Pre-Processing

### Corpus Description

The corpus description can be found [here](https://docsouth.unc.edu/docsouthdata/).

### Import Data

Read in all of the .txt files in two folders, do some pre-processing on it, and concat them all into a Pandas dataframe

In [6]:
meta_fpn = pandas.read_csv('/Users/jamescody/GitHubCode/measuring_intersectionality/data/first-person-narratives-american-south/data/toc.csv', encoding = 'utf-8')
meta_neh = pandas.read_csv('/Users/jamescody/GitHubCode/measuring_intersectionality/data/na-slave-narratives/data/toc.csv', encoding = 'utf-8')
meta = pandas.concat([meta_fpn, meta_neh]).reset_index()

In [7]:
meta_fpn.shape

(150, 7)

In [8]:
meta_neh.shape

(294, 7)

In [9]:
meta_fpn

,Filename,Author,Gender,Title,Date,URL,URL(text-only)
0,fpn-crumpton-crumpton.xml,H. J. Crumpton,m,The Adventures of Two Alabama Boys,1912.0,http://docsouth.unc.edu/fpn/crumpton/menu.html,http://docsouth.unc.edu/full-text/first-person...
1,fpn-caldwell-caldwell.xml,Joseph Caldwell,m,Autobiography and Biography of Rev. Joseph Cal...,1860.0,http://docsouth.unc.edu/fpn/caldwell/menu.html,http://docsouth.unc.edu/full-text/first-person...
2,fpn-carroll-carroll.xml,John W. Carroll,m,Autobiography and Reminiscences of John W. Car...,1898.0,http://docsouth.unc.edu/fpn/carroll/menu.html,http://docsouth.unc.edu/full-text/first-person...
3,fpn-biggs-biggs.xml,Asa Biggs,m,"Autobiography of Asa Biggs, Including a Journa...",1915.0,http://docsouth.unc.edu/fpn/biggs/menu.html,http://docsouth.unc.edu/full-text/first-person...
4,fpn-lane-lane.xml,Isaac Lane,m,"Autobiography of Bishop Isaac Lane, LL.D. with...",1916.0,http://docsouth.unc.edu/fpn/lane/menu.html,http://docsouth.unc.edu/full-text/first-person...
...,...,...,...,...,...,...,...
145,fpn-wyeth-wyeth.xml,John A. Wyeth,m,With Sabre and Scalpel; the Autobiography of a...,1914.0,http://docsouth.unc.edu/fpn/wyeth/menu.html,http://docsouth.unc.edu/full-text/first-person...
146,fpn-velazquez-velazquez.xml,Loreta Janeta Velazquez,f,The Woman in Battle: A Narrative of the Exploi...,1876.0,http://docsouth.unc.edu/fpn/velazquez/menu.html,http://docsouth.unc.edu/full-text/first-person...
147,fpn-pringle-pringle.xml,Elizabeth Waties Allston Pringle,f,A Woman Rice Planter,1914.0,http://docsouth.unc.edu/fpn/pringle/menu.html,http://docsouth.unc.edu/full-text/first-person...
148,fpn-collis-collis.xml,Septima M. Collis,f,"A Woman's War Record, 1861-1865",1889.0,http://docsouth.unc.edu/fpn/collis/menu.html,http://docsouth.unc.edu/full-text/first-person...


In [10]:
#Dropping multiple editions from the same autobiography
#Keeping the autobiography with the latest date

#349 and 270 = Frederick Douglass
#363 = William Wells Brown
meta.drop([349, 270, 363], inplace=True)

meta.drop_duplicates(subset = 'Filename', inplace=True)

In [11]:
#read in all the data, with some cleaning
path_fpn = get_path('/Users/jamescody/GitHubCode/measuring_intersectionality/data/first-person-narratives-american-south/data/texts/') # indicate the local path where files are stored
path_neh = get_path('/Users/jamescody/GitHubCode/measuring_intersectionality/data/first-person-narratives-american-south/data/texts/')
path_all = path_fpn + path_neh


In [12]:
path_all

['/Users/jamescody/GitHubCode/measuring_intersectionality/data/first-person-narratives-american-south/data/texts/fpn-velazquez-velazquez.txt',
 '/Users/jamescody/GitHubCode/measuring_intersectionality/data/first-person-narratives-american-south/data/texts/fpn-ferebee-ferebee.txt',
 '/Users/jamescody/GitHubCode/measuring_intersectionality/data/first-person-narratives-american-south/data/texts/fpn-wood-wood.txt',
 '/Users/jamescody/GitHubCode/measuring_intersectionality/data/first-person-narratives-american-south/data/texts/fpn-clinkscales-clinksc.txt',
 '/Users/jamescody/GitHubCode/measuring_intersectionality/data/first-person-narratives-american-south/data/texts/fpn-mckim-mckim.txt',
 '/Users/jamescody/GitHubCode/measuring_intersectionality/data/first-person-narratives-american-south/data/texts/fpn-robinson-robinson.txt',
 '/Users/jamescody/GitHubCode/measuring_intersectionality/data/first-person-narratives-american-south/data/texts/fpn-negpeon-negpeon.txt',
 '/Users/jamescody/GitHubCo

In [13]:
#path_work = get_path('/Users/jamescody/Documents/_Dissertation/Topics/')

In [14]:
#path_work

In [15]:
#remove duplicate files and multiple editions of same narrative
keep = meta['Filename'].tolist()
keep = [name.replace('.xml', '.txt') for name in keep]
filenames = []
path = []

In [16]:
for p in path_all:
    if (p.split('/')[-1] not in filenames) and (p.split('/')[-1] in keep):
        filenames.append(p.split('/')[-1])
        path.append(p)
    else:
        pass

In [ ]:
path

This outs all the data in one file

In [ ]:
data = []

for file in path:
    with open(file, encoding='utf-8') as myfile:
        data.append(myfile.read())

### Pre-Processing

Word2Vec learns about the relationships among words by observing them in context. This means that we want to split our texts into word-units. In this text there is no punctuation, and thus nothing resembling a sentence. In other text we  want to maintain sentence boundaries as well, since the last word of the previous sentence might skew the meaning of the next sentence.

You can split your text in sentences using ` nltk.tokenize.sent_tokenize()`

In [19]:
sentences = [sentence for text in data for sentence in sent_tokenize(text)]
words_by_sentence = [fast_tokenize(sentence) for sentence in sentences]
words_by_sentence = [sentence for sentence in words_by_sentence if sentence != []]
words_by_sentence[0]

['madam', 'velasquez', 'in', 'female', 'attire']

In [22]:
model = gensim.models.Word2Vec(words_by_sentence, #size=100, 
                               window=5,
                               min_count=10, sg=1, alpha=0.025, #iter=5, 
                               batch_words=10000, workers=1)

# Save model for later use
model.wv.save_word2vec_format('/Users/jamescody/GitHubCode/measuring_intersectionality/data/word2vec_all_clean.txt')

In [24]:
#create 40 random models for constructing confidence intervals

def gen_model(words_by_sentence, num):
    """
    Takes a list of words by senence as input and a number (for naming the file)
    Saves a word2vec model in the word2vec_robust folder
    """

    model = gensim.models.Word2Vec(words_by_sentence, #size=100, 
                                   window=5,
                                   min_count=10, sg=1, alpha=0.025, #iter=5, 
                                   batch_words=10000, workers=1)
    
    
    model.wv.save_word2vec_format('/Users/jamescody/GitHubCode/measuring_intersectionality/data/word2vec_robust/model%d.txt' % num)
    
#Number of sentences, for use in creating random sentences
num_sent = len(words_by_sentence)

for num in range(0,40):
    print(num)
    
    #extract random sample of sentences with replacement, 
    #equal to total number of sentences in the full corpus
    gen_model(choices(words_by_sentence, k = num_sent), num)

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
